# Preprocessing

## 1. Align visits

Align your data into visit based data, with unique identifier `ID` and timestamps `date` preserved. 

## 2. Process data into input tensors

```
import DP as dp
processed_data, time_info, missing, masking = dp.partition_multi_seq(real_df, unique_identifier, maximum_visits)
```

`processed_data` is a data with the first dimension as the number of samples $N$, and second dimension as the maximum visit $T$, and the third dimension is the features $F$

`time_info` is a embeded timestamps, with dimension $(N,T)$

`missing` is a masking tensor for missing data $(N,T,F)$

`masking` is a masking tensor for paddings $(N,T,F)$

Example data:

In [7]:
import os
import numpy as np
import pandas as pd
from model import DP as dp

# --- reproducibility (optional) ---
rng = np.random.default_rng(42)

# --- simulate features ---
# 100,000 samples, 10-dimensional MVN(0, I)
variables = rng.multivariate_normal(mean=np.zeros(10), cov=np.eye(10), size=100_000)
vars_df = pd.DataFrame(variables, columns=[f"x{i+1}" for i in range(10)])

# --- ids and dates ---
patient_id = rng.integers(low=0, high=1000, size=100_000)
# random dates within ~27.4 years after 2000-01-01
date = np.datetime64("2000-01-01") + rng.integers(low=0, high=10_000, size=100_000).astype("timedelta64[D]")

# build base frame
data = pd.DataFrame({"patient_id": patient_id, "date": pd.to_datetime(date)})

# concatenate columns (align by row index)
data = pd.concat([data.reset_index(drop=True), vars_df.reset_index(drop=True)], axis=1)
data.sort_values(['patient_id','date']).reset_index(drop=True,inplace=True)

# --- downstream processing ---
processed_data, time_info, missing, masking = dp.partition_multi_seq(data, column_to_partition="patient_id", threshold=1, max_len=100)


100%|██████████| 11/11 [00:00<00:00, 89.37it/s]

['patient_id', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10']
torch.Size([1, 100000, 11])
torch.Size([1, 100000, 11])
Number of bins:0, Number of nums: 11, Number of cats:0


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]